<div align="center">

# 🚕 Yandex Go Analytics

## 4 · Data Cleaning

*Turn raw Bronze records into trustworthy Silver tables.*

![layer](https://img.shields.io/badge/layer-silver-9E9E9E)
![engine](https://img.shields.io/badge/engine-Structured%20Streaming-E25A1C)
![quality](https://img.shields.io/badge/quality-validated-2E7D32)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [5 · Aggregations & Optimizations](./5.%20Aggregations%20%26%20Optimizations.ipynb) |

**Progress** &nbsp; ▰▰▰▰▱▱▱▱  **4 / 8**

---

## 🎯 Task requirements

Read from Bronze, apply the transformations below and save the results to the `silver` schema.

### 🚕 `silver.taxi`
- [x] Keep rows with `passenger_count > 0`, `trip_distance > 0`, `fare_amount > 0`
- [x] Map `payment_type` to a new description column: `1 = Credit Card`, `2 = Cash`, otherwise `Other`
- [x] Add an ingestion timestamp (`current_timestamp()`) and drop duplicates

### 🧑‍✈️ `silver.drivers`
- [x] Keep rows with `experience > 0` and `rating` between `1.0` and `5.0`
- [x] Add an ingestion timestamp and drop duplicates by the driver's ID

### 👤 `silver.users`
- [x] Filter out records with a null name
- [x] Replace nulls in the second business attribute with `"Unknown"`
- [x] Add an ingestion timestamp and drop duplicates

## 🧩 How it is solved

| Table | Filter | Transformation | De-duplication key |
|:--|:--|:--|:--|
| `silver.taxi` | `passenger_count > 0` ∧ `trip_distance > 0` ∧ `fare_amount > 0` | `payment_type_description` (`CASE WHEN`), `_processed_at` | `user_id`, `driver_id`, `pickup_datetime`, `dropoff_datetime` |
| `silver.drivers` | `experience > 0` ∧ `1.0 ≤ rating ≤ 5.0` | `_processed_at` | `id` |
| `silver.users` | `full_name IS NOT NULL` | `email` → `COALESCE(email, 'Unknown')`, `_processed_at` | `email` |

> 📝 The ingestion timestamp column is named `_processed_at`; the users' schema has `full_name` / `email`
> (see the *naming notes* in the project overview).

Every service reads its Bronze table as a **Delta stream** (`ignoreChanges`) and appends to Silver with
`availableNow` + a checkpoint — so only *new* Bronze rows are processed on every run.

```text
 bronze.taxi    ──▶ TripSilverTransformationService   ──▶ silver.taxi
 bronze.drivers ──▶ DriverSilverTransformationService ──▶ silver.drivers
 bronze.users   ──▶ UserSilverTransformationService   ──▶ silver.users
```

## 1️⃣ Connect to Spark

In [7]:
from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-data-cleaning")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 13:49:28.675 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-data-cleaning | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_dev | env = env


## 2️⃣  Run the Silver transformations

```text
 filter  ─▶  decode / enrich  ─▶  add _processed_at  ─▶  drop duplicates  ─▶  append to Silver
```

In [8]:
from src.core.etl.silver import (
    DriverSilverTransformationService,
    TripSilverTransformationService,
    UserSilverTransformationService,
)

print("1. bronze.taxi    ->  silver.taxi")
TripSilverTransformationService().run(spark)

print("\n2. bronze.drivers ->  silver.drivers")
DriverSilverTransformationService().run(spark)

print("\n3. bronze.users   ->  silver.users")
UserSilverTransformationService().run(spark)

print("\n✅ Silver layer is up to date")

2026-10-09 13:49:28.683 | INFO     | src.core.etl.silver.base:run:100 - Starting incremental transformation: yandex_go_dev.bronze.taxi -> yandex_go_dev.silver.taxi
2026-10-09 13:49:28.684 | INFO     | src.core.etl.silver.base:extract:54 - Reading incremental stream from Bronze: yandex_go_dev.bronze.taxi
2026-10-09 13:49:28.686 | INFO     | src.core.etl.silver.base:load:80 - Writing incremental batch to Silver: yandex_go_dev.silver.taxi


1. bronze.taxi    ->  silver.taxi


StreamingQueryException: [STREAM_FAILED] Query [id = 9e93e88c-49bb-4df4-9ed7-f31c093a175c, runId = 2fd73e9f-c8a5-4a8b-a406-0d4f2c1a0d69] terminated with exception: [STATE_STORE_CHECKPOINT_LOCATION_NOT_EMPTY] The checkpoint location dbfs:/Volumes/yandex_go_dev/raw_files/landing/_checkpoints/silver/taxi/data/state should be empty on batch 0
Please either use a new checkpoint location, or delete the existing data in the checkpoint location. SQLSTATE: 42K03 SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = 9e93e88c-49bb-4df4-9ed7-f31c093a175c, runId = 2fd73e9f-c8a5-4a8b-a406-0d4f2c1a0d69]
Current Start Offsets: {}
Current End Offsets: {}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSourceV1 `yandex_go_dev`.`silver`.`taxi`, DeltaSink[s3://dbstorage-prod-siapzyrqgb/uc/eb1e0466-1e4a-4fcd-b424-0139b069b235/668f049e-6779-4567-b46a-4f843624c518/__unitystorage/catalogs/eaefdaec-3435-4282-9a88-837d980583de/tables/34f6f263-cc15-4df1-8e97-1a213390ea86], 

---

## ✅ Checkpoint

- [x] `silver.taxi` — invalid trips removed, payment type decoded, duplicates dropped
- [x] `silver.drivers` — valid experience and rating only, one row per driver
- [x] `silver.users` — named users only, missing e-mail replaced by `Unknown`, duplicates dropped

> **Next up ▶ [5 · Aggregations & Optimizations](./5.%20Aggregations%20%26%20Optimizations.ipynb)** — aggregate Silver into the Gold metrics and optimise the tables.

<div align="center"><sub>Yandex Go Analytics · notebook 4 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>